# 04 — Modelo 2: Collaborative Filtering con ALS
## Cross-Selling Recommender | Proyecto Final Henry

**Por qué ALS sobre SVD:**
Instacart no tiene ratings explícitos. ALS modela feedback implícito
(frecuencia de compra) con la fórmula C_ui = 1 + alpha × frecuencia_ui.
Con sparsity > 99.9%, ALS supera a SVD según Hu et al. (2008).


In [ ]:
# ── 1. Configuración ────────────────────────────────────────────────────────
import sys, logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

PROJECT_ROOT = Path().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_config, load_instacart_tables, build_master_dataframe
from src.data.preprocessor import run_cleaning_pipeline
from src.models.collaborative_filter import (
    build_implicit_matrix, train_als,
    get_recommendations_for_user, get_similar_products, save_als_model,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


---
## 2. Carga y Preprocesamiento

In [ ]:
# ── 2.1 Cargar tablas ────────────────────────────────────────────────────────
cfg = load_config()
orders, op_prior, products, aisles, departments = load_instacart_tables(cfg)

df_master = build_master_dataframe(
    orders=orders, order_products=op_prior,
    products=products, aisles=aisles,
    departments=departments, eval_set='prior',
)
product_names = products.set_index('product_id')['product_name'].to_dict()
dept_map = (
    products.merge(departments, on='department_id', how='left')
    .set_index('product_id')['department'].to_dict()
)
print(f'DataFrame maestro: {df_master.shape}')


In [ ]:
# ── 2.2 Pipeline de limpieza — df_full para ALS ──────────────────────────────
# ALS usa el dataset COMPLETO (no el subsample de 200K)
# Más usuarios y compras → mejores factores latentes
df_full, _ = run_cleaning_pipeline(
    df=df_master,
    min_basket_items=2,
    add_temporal=True,
    apriori_sample=200_000,
    random_state=RANDOM_STATE,
)
print(f'df_full (para ALS): {df_full.shape}')
print(f'Usuarios únicos   : {df_full["user_id"].nunique():,}')
print(f'Productos únicos  : {df_full["product_id"].nunique():,}')


---
## 3. Construcción de la Matriz de Confianza

`C_ui = 1 + alpha × frecuencia_ui` (Hu et al., 2008). alpha=40 amplifica compras repetidas.

In [ ]:
# ── 3.1 Matriz item × user (formato que espera implicit) ─────────────────────
item_user_matrix, user_ids, product_ids = build_implicit_matrix(df=df_full, alpha=40.0)
sparsity = 1 - item_user_matrix.nnz / (item_user_matrix.shape[0] * item_user_matrix.shape[1])
print(f'Forma: {item_user_matrix.shape[0]:,} ítems × {item_user_matrix.shape[1]:,} usuarios')
print(f'Entradas no cero: {item_user_matrix.nnz:,}')
print(f'Sparsity: {sparsity:.4%}')
print('Alta sparsity valida ALS sobre SVD: modela incertidumbre en los ceros.')


In [ ]:
# ── 3.2 Distribución de interacciones por usuario ────────────────────────────
ipp = np.diff(item_user_matrix.T.tocsr().indptr)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ipp, bins=50, color='#4C72B0', alpha=0.85, edgecolor='white')
ax.axvline(np.median(ipp), color='red', linestyle='--', linewidth=1.5,
           label=f'Mediana: {np.median(ipp):.0f}')
ax.set_xlabel('Productos comprados por usuario'); ax.set_ylabel('Usuarios')
ax.set_title('Distribución de interacciones por usuario'); ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '12_cf_interactions_per_user.png', bbox_inches='tight')
plt.show()


---
## 4. Entrenamiento ALS

In [ ]:
# ── 4.1 Entrenar ALS ─────────────────────────────────────────────────────────
# factors=50    : balance expresividad/overfitting
# iterations=20 : pérdida estabiliza antes de iter 15
# reg=0.01      : L2 estándar
model = train_als(
    item_user_matrix=item_user_matrix,
    factors=50, iterations=20, regularization=0.01,
    random_state=RANDOM_STATE,
)
print(f'Factores latentes  : {model.factors}')
print(f'Iteraciones        : {model.iterations}')
print(f'item_factors shape : {model.item_factors.shape}')
print(f'user_factors shape : {model.user_factors.shape}')


---
## 5. Recomendaciones por Usuario

In [ ]:
# ── 5.1 Top-10 para 3 usuarios de ejemplo ────────────────────────────────────
for uid in user_ids[:3]:
    recs = get_recommendations_for_user(
        user_id=uid, model=model, user_ids=user_ids, product_ids=product_ids,
        item_user_matrix=item_user_matrix, product_names=product_names, top_k=10,
    )
    print(f'Usuario {uid}:')
    print(recs.to_string(index=False) if not recs.empty else '  Sin recomendaciones.')
    print()


---
## 6. Productos Similares (Item-to-Item)

In [ ]:
# ── 6.1 Similitud coseno en espacio latente ───────────────────────────────────
for pid in product_ids[:3]:
    sim = get_similar_products(
        product_id=pid, model=model, product_ids=product_ids,
        product_names=product_names, top_k=6,
    )
    pname = product_names.get(pid, f'ID {pid}')
    print(f'Similares a: {pname}')
    print(sim.to_string(index=False) if not sim.empty else '  Sin similares.')
    print()


---
## 7. Espacio Latente — PCA 2D

In [ ]:
# ── 7.1 PCA de factores de productos coloreado por departamento ───────────────
n_viz = min(2000, len(product_ids))
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(len(product_ids), n_viz, replace=False)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(model.item_factors[idx])
ev = pca.explained_variance_ratio_.sum() * 100
print(f'Varianza explicada (2 PCs): {ev:.1f}%')

sample_pids = [product_ids[i] for i in idx]
sample_depts = [dept_map.get(p, 'other') for p in sample_pids]
df_pca = pd.DataFrame({'x': coords[:,0], 'y': coords[:,1], 'dept': sample_depts})
top_d = df_pca['dept'].value_counts().head(10).index.tolist()

fig, ax = plt.subplots(figsize=(11, 7))
for d in top_d:
    s = df_pca[df_pca['dept'] == d]
    ax.scatter(s['x'], s['y'], s=12, alpha=0.5, label=d)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('Espacio latente ALS — Productos por departamento (PCA 2D)', fontsize=13, fontweight='bold')
ax.legend(title='Departamento', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '13_cf_latent_space_pca.png', bbox_inches='tight')
plt.show()
print('Guardado: outputs/figures/13_cf_latent_space_pca.png')


---
## 8. Guardado del Modelo

In [ ]:
# ── 8.1 Guardar modelo ALS + mapeos ──────────────────────────────────────────
path = save_als_model(model=model, user_ids=user_ids, product_ids=product_ids, filename='als_model.pkl')
print(f'Modelo guardado: {path}')
print(f'Usuarios cubiertos : {len(user_ids):,}')
print(f'Productos cubiertos: {len(product_ids):,}')
print(f'Tamaño archivo     : {path.stat().st_size / 1024 / 1024:.1f} MB')
print('\n-> Siguiente: notebook 05_evaluation.ipynb')
